<a href="https://colab.research.google.com/github/Sylva-gif/Licences-LUS/blob/main/Saprk_TP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
# Install pyspark if not already installed
if 'pyspark' not in sys.modules:
    !pip install pyspark

## Partie I : Traitement de texte avec RDD

### Step 1 & 2 : Télécharger la dataset « Shakespeare » et créer un nouveau notebook Jupyter Python

Pour cette partie, nous allons supposer que vous avez déjà téléchargé le fichier `shakespeare.txt` et que vous l'avez rendu disponible dans votre environnement de travail (par exemple, en l'uploadant dans le système de fichiers de Colab). Le Jupyter Notebook est déjà créé, donc nous pouvons passer directement à la lecture du fichier.

### Step 3 : Lire le fichier et le stocker sous forme de RDD en utilisant l’opération « textFile »

L'opération `textFile` est une *transformation* qui lit un fichier texte depuis HDFS, un système de fichiers local (comme dans Colab), ou toute autre source supportée par Hadoop, et retourne un RDD de chaînes de caractères où chaque élément est une ligne du fichier.

La variable `sc` représente le `SparkContext`, qui est le point d'entrée pour toutes les fonctionnalités Spark. Il est nécessaire pour créer des RDDs.

Nous allons ensuite vérifier que le fichier a été chargé avec succès en appelant la méthode `count()` pour afficher le nombre de lignes dans le RDD.

In [2]:
from pyspark import SparkContext

# Initialiser SparkContext si ce n'est pas déjà fait
# Dans Colab, SparkContext est souvent déjà initialisé avec pyspark.sql.SparkSession
# Nous allons tenter de le récupérer ou le créer.
try:
    sc = SparkContext.getOrCreate()
except ValueError:
    sc = SparkContext(appName="ShakespeareRDD")

# NOTE: Assurez-vous que 'shakespeare.txt' est disponible dans votre environnement Colab.
# Par exemple, vous pouvez l'uploader manuellement ou le télécharger.
# Pour l'exemple, nous allons créer un fichier dummy si 'shakespeare.txt' n'existe pas.
import os
if not os.path.exists('shakespeare.txt'):
    with open('shakespeare.txt', 'w') as f:
        f.write("To be or not to be, that is the question:\n")
        f.write("Whether 'tis nobler in the mind to suffer\n")
        f.write("The slings and arrows of outrageous fortune,\n")
        f.write("Or to take arms against a sea of troubles\n")
        f.write("And by opposing end them.\n")
    print("Fichier 'shakespeare.txt' créé pour l'exemple.")

# Lire le fichier et le stocker sous forme de RDD
lines_rdd = sc.textFile("shakespeare.txt")

# Vérifier le nombre d'éléments dans le RDD
print(f"Le nombre de lignes dans le RDD est : {lines_rdd.count()}")

Fichier 'shakespeare.txt' créé pour l'exemple.
Le nombre de lignes dans le RDD est : 5


### Step 4 : Diviser chaque ligne en des mots

Nous allons maintenant diviser chaque ligne de notre RDD `lines_rdd` en un ensemble de mots. Pour ce faire, nous utiliserons la méthode `flatMap` et une fonction anonyme `lambda` qui scindera chaque ligne par les espaces. Nous allons également convertir les mots en minuscules et supprimer la ponctuation pour un traitement ultérieur plus propre.

In [3]:
import re

# Fonction pour nettoyer et diviser les lignes en mots
def split_words(line):
    # Convertir en minuscules et supprimer la ponctuation, puis diviser par les espaces
    return re.findall(r'\b\w+\b', line.lower())

# Appliquer flatMap pour obtenir un RDD de mots
words_rdd = lines_rdd.flatMap(split_words)

# Afficher les 10 premiers mots pour vérifier
print("Les 10 premiers mots sont:", words_rdd.take(10))

# Compter le nombre total de mots
print(f"Le nombre total de mots est : {words_rdd.count()}")

Les 10 premiers mots sont: ['to', 'be', 'or', 'not', 'to', 'be', 'that', 'is', 'the', 'question']
Le nombre total de mots est : 39


### Step 5 : Extraire sous forme clé, valeur

Pour chaque mot que nous avons extrait, nous allons maintenant lui attribuer la valeur `1` par défaut. Ceci est une étape courante dans le processus de comptage de mots, où chaque occurrence d'un mot est initialement comptée comme 1. Nous utiliserons la transformation `map` pour créer ces paires (mot, 1).

In [4]:
# Attribuer la valeur 1 à chaque mot pour créer des paires (mot, 1)
word_counts_rdd = words_rdd.map(lambda word: (word, 1))

# Afficher les 10 premières paires pour vérifier
print("Les 10 premières paires (mot, 1) sont:", word_counts_rdd.take(10))

Les 10 premières paires (mot, 1) sont: [('to', 1), ('be', 1), ('or', 1), ('not', 1), ('to', 1), ('be', 1), ('that', 1), ('is', 1), ('the', 1), ('question', 1)]


### Step 6 : Reduce : faire la somme des valeurs pour chaque mot

Maintenant que nous avons des paires `(mot, 1)`, nous pouvons utiliser la méthode `reduceByKey()` pour agréger les comptes pour chaque mot. Cette méthode prend une fonction d'agrégation (ici, l'addition) qui sera appliquée à toutes les valeurs associées à une même clé (le mot). Le résultat sera un RDD de paires `(mot, count)` où `count` est le nombre total d'apparitions de ce mot.

In [5]:
# Utiliser reduceByKey pour sommer les valeurs pour chaque mot
word_counts_summed_rdd = word_counts_rdd.reduceByKey(lambda a, b: a + b)

# Afficher les 10 premiers mots avec leurs comptes
print("Les 10 premiers mots avec leurs comptes sont:", word_counts_summed_rdd.take(10))

# Afficher le compte total de mots uniques
print(f"Le nombre de mots uniques est : {word_counts_summed_rdd.count()}")

Les 10 premiers mots avec leurs comptes sont: [('to', 4), ('question', 1), ('nobler', 1), ('suffer', 1), ('slings', 1), ('and', 2), ('arrows', 1), ('of', 2), ('fortune', 1), ('take', 1)]
Le nombre de mots uniques est : 30


### Step 7 : Écrire le résultat dans un fichier texte dans votre disque avec `coalesce(1).saveAsTextFile()`

Pour écrire le RDD final contenant les comptes de mots dans un fichier texte, nous utilisons la méthode `saveAsTextFile()`. Pour s'assurer que toutes les partitions du RDD sont écrites dans un seul fichier de sortie, nous utilisons `coalesce(1)` pour regrouper toutes les données dans une seule partition avant de les sauvegarder. Spark créera un dossier avec le nom spécifié et y placera le fichier de sortie (généralement nommé `part-00000`).

In [6]:
# Définir le répertoire de sortie
output_directory = "shakespeare_word_counts"

# Supprimer le répertoire de sortie s'il existe déjà pour éviter les erreurs
import shutil
if os.path.exists(output_directory):
    shutil.rmtree(output_directory)
    print(f"Répertoire '{output_directory}' existant supprimé.")

# Écrire le RDD des comptes de mots dans un fichier texte
# coalesce(1) assure que toutes les données sont écrites dans un seul fichier
word_counts_summed_rdd.coalesce(1).saveAsTextFile(output_directory)

print(f"Les comptes de mots ont été sauvegardés dans le répertoire '{output_directory}'.")

# Optionnel: Lire le contenu du fichier pour vérifier
# Vous devrez lister les fichiers dans le répertoire et lire le fichier 'part-00000'
# Pour cela, nous pouvons utiliser un peu de code Python standard après la sauvegarde
import glob
output_files = glob.glob(f"{output_directory}/part-*")
if output_files:
    print(f"Contenu du fichier de sortie '{output_files[0]}':")
    with open(output_files[0], 'r') as f:
        for i, line in enumerate(f):
            print(line.strip())
            if i >= 9: # Afficher les 10 premières lignes
                break
else:
    print("Aucun fichier de sortie trouvé.")

Les comptes de mots ont été sauvegardés dans le répertoire 'shakespeare_word_counts'.
Contenu du fichier de sortie 'shakespeare_word_counts/part-00000':
('to', 4)
('question', 1)
('nobler', 1)
('suffer', 1)
('slings', 1)
('and', 2)
('arrows', 1)
('of', 2)
('fortune', 1)
('take', 1)


---

## Partie I terminée.

Nous allons maintenant passer à la **Partie II : Exploration des données avec Spark (Manipulation d’un Dataframe)**. Cette partie utilisera la base de données `daily_weather.csv`.

### Partie II : Exploration des données avec Spark

### Step 1 : Créer une SparkSession et charger les données dans un Spark DataFrame

Nous allons d'abord importer la classe `SparkSession` de `pyspark.sql`, puis créer une instance de `SparkSession`. Cette instance est le point d'entrée pour programmer Spark avec l'API DataFrame et SQL. Ensuite, nous chargerons le fichier `daily_weather.csv` dans un DataFrame. Nous utiliserons `inferSchema=True` pour que Spark déduise automatiquement les types de colonnes, et `header=True` car le fichier contient une ligne d'en-tête.

In [7]:
from pyspark.sql import SparkSession
import os

# Crée une SparkSession
spark = SparkSession.builder.appName("SparkExample").getOrCreate()

# NOTE: Assurez-vous que 'daily_weather.csv' est disponible dans votre environnement Colab.
# Par exemple, vous pouvez l'uploader manuellement ou le télécharger.
# Pour l'exemple, nous allons créer un fichier dummy si 'daily_weather.csv' n'existe pas.
if not os.path.exists('daily_weather.csv'):
    # Créer un fichier CSV dummy pour l'exemple
    dummy_data = "air_temp_9am,rain_duration_9am,rain_accumulation_9am,air_pressure_9am\n20.5,10,5,1012.5\n22.0,0,0,1011.0\n18.3,20,10,1013.2\n25.1,0,0,1010.0\n19.8,5,2.5,1014.1\n"
    with open('daily_weather.csv', 'w') as f:
        f.write(dummy_data)
    print("Fichier 'daily_weather.csv' créé pour l'exemple.")

# Charge les données dans un DataFrame
df = spark.read.format("csv").load("daily_weather.csv", header=True, inferSchema=True)

# Pour afficher les noms des colonnes
print("Noms des colonnes:", df.columns)

# Pour vérifier le schéma inféré
print("Schéma du DataFrame:")
df.printSchema()

print("Affichage des premières lignes du DataFrame:")
df.show(5)

Fichier 'daily_weather.csv' créé pour l'exemple.
Noms des colonnes: ['air_temp_9am', 'rain_duration_9am', 'rain_accumulation_9am', 'air_pressure_9am']
Schéma du DataFrame:
root
 |-- air_temp_9am: double (nullable = true)
 |-- rain_duration_9am: integer (nullable = true)
 |-- rain_accumulation_9am: double (nullable = true)
 |-- air_pressure_9am: double (nullable = true)

Affichage des premières lignes du DataFrame:
+------------+-----------------+---------------------+----------------+
|air_temp_9am|rain_duration_9am|rain_accumulation_9am|air_pressure_9am|
+------------+-----------------+---------------------+----------------+
|        20.5|               10|                  5.0|          1012.5|
|        22.0|                0|                  0.0|          1011.0|
|        18.3|               20|                 10.0|          1013.2|
|        25.1|                0|                  0.0|          1010.0|
|        19.8|                5|                  2.5|          1014.1|
+-----

### Step 2 : Visualisation des colonnes et leurs types avec les méthodes `.columns` et `printSchema()`

1.  **Interprétation du résultat** : Les résultats de `df.columns` et `df.printSchema()` dans l'étape précédente nous ont déjà montré les noms des colonnes et leurs types respectifs. `air_temp_9am`, `rain_duration_9am`, `rain_accumulation_9am` et `air_pressure_9am` sont tous de type `double`, ce qui est approprié pour des mesures numériques.

2.  **Afficher les 5 premiers éléments avec `head(5)`** : Nous allons maintenant afficher les cinq premières lignes du DataFrame pour avoir un aperçu des données.

In [8]:
# Afficher les 5 premiers éléments du DataFrame
print("Les 5 premières lignes du DataFrame (avec head(5)):")
print(df.head(5))

# Pour une meilleure lisibilité, nous pouvons aussi utiliser .show(5)
print("Les 5 premières lignes du DataFrame (avec show(5)):")
df.show(5)

Les 5 premières lignes du DataFrame (avec head(5)):
[Row(air_temp_9am=20.5, rain_duration_9am=10, rain_accumulation_9am=5.0, air_pressure_9am=1012.5), Row(air_temp_9am=22.0, rain_duration_9am=0, rain_accumulation_9am=0.0, air_pressure_9am=1011.0), Row(air_temp_9am=18.3, rain_duration_9am=20, rain_accumulation_9am=10.0, air_pressure_9am=1013.2), Row(air_temp_9am=25.1, rain_duration_9am=0, rain_accumulation_9am=0.0, air_pressure_9am=1010.0), Row(air_temp_9am=19.8, rain_duration_9am=5, rain_accumulation_9am=2.5, air_pressure_9am=1014.1)]
Les 5 premières lignes du DataFrame (avec show(5)):
+------------+-----------------+---------------------+----------------+
|air_temp_9am|rain_duration_9am|rain_accumulation_9am|air_pressure_9am|
+------------+-----------------+---------------------+----------------+
|        20.5|               10|                  5.0|          1012.5|
|        22.0|                0|                  0.0|          1011.0|
|        18.3|               20|               

### Step 3 : Afficher les statistiques récapitulatives

Nous pouvons afficher les statistiques récapitulatives pour toutes les colonnes numériques du DataFrame en utilisant la méthode `describe()`. Cette méthode calcule des statistiques comme le compte, la moyenne, l'écart type, le minimum et le maximum. Pour une meilleure visualisation, nous utiliserons `.show()`.

Nous allons aussi afficher les statistiques récapitulatives pour une seule colonne, par exemple `air_temp_9am`, pour illustrer cette fonctionnalité.

In [9]:
# Afficher les statistiques récapitulatives pour toutes les colonnes
print("Statistiques récapitulatives pour toutes les colonnes:")
df.describe().show()

# Afficher les statistiques récapitulatives pour une seule colonne (par exemple, 'air_temp_9am')
print("Statistiques récapitulatives pour la colonne 'air_temp_9am':")
df.describe('air_temp_9am').show()

# La méthode .toPandas() est mentionnée dans l'exercice pour convertir le DataFrame Spark en DataFrame Pandas
# pour une visualisation ou manipulation potentiellement plus facile si nécessaire.
# Exemple (non requis par l'étape 3, mais pour référence):
# pandas_df = df.describe().toPandas()
# print("Statistiques récapitulatives converties en Pandas DataFrame (premières lignes):")
# print(pandas_df.head())

Statistiques récapitulatives pour toutes les colonnes:
+-------+------------------+-----------------+---------------------+------------------+
|summary|      air_temp_9am|rain_duration_9am|rain_accumulation_9am|  air_pressure_9am|
+-------+------------------+-----------------+---------------------+------------------+
|  count|                 5|                5|                    5|                 5|
|   mean|             21.14|              7.0|                  3.5|1012.1600000000001|
| stddev|2.5832150510555647|8.366600265340756|    4.183300132670378|1.6562004709575773|
|    min|              18.3|                0|                  0.0|            1010.0|
|    max|              25.1|               20|                 10.0|            1014.1|
+-------+------------------+-----------------+---------------------+------------------+

Statistiques récapitulatives pour la colonne 'air_temp_9am':
+-------+------------------+
|summary|      air_temp_9am|
+-------+------------------+
|  c

### Step 4 : Création de nouvelles colonnes et calculs agrégés

1.  **Créer un nouveau DataFrame avec une colonne qui s’appelle « ratio » qui calcule le taux de `rain_duration_9am / rain_accumulation_9am`**

    Nous allons utiliser la méthode `withColumn()` pour ajouter cette nouvelle colonne au DataFrame. Il est important de noter que si `rain_accumulation_9am` est zéro, la division entraînera un résultat `Infinity` ou `NaN` (Not a Number), ce qui est un comportement attendu en Spark SQL pour les divisions par zéro.

In [12]:
from pyspark.sql.functions import col, try_divide

# Créer la colonne 'ratio'
# Utiliser try_divide pour gérer les divisions par zéro qui produiront des NULL
df1 = df.withColumn("ratio", try_divide(col("rain_duration_9am"), col("rain_accumulation_9am")))

# Afficher le contenu du nouveau DataFrame avec la colonne 'ratio'
print("DataFrame avec la nouvelle colonne 'ratio':")
df1.show()

DataFrame avec la nouvelle colonne 'ratio':
+------------+-----------------+---------------------+----------------+-----+
|air_temp_9am|rain_duration_9am|rain_accumulation_9am|air_pressure_9am|ratio|
+------------+-----------------+---------------------+----------------+-----+
|        20.5|               10|                  5.0|          1012.5|  2.0|
|        22.0|                0|                  0.0|          1011.0| NULL|
|        18.3|               20|                 10.0|          1013.2|  2.0|
|        25.1|                0|                  0.0|          1010.0| NULL|
|        19.8|                5|                  2.5|          1014.1|  2.0|
+------------+-----------------+---------------------+----------------+-----+



2.  **Afficher le contenu de nouvelle colonne `ratio`**

    Nous allons sélectionner spécifiquement la colonne `ratio` pour n'afficher que ses valeurs.

In [13]:
# Afficher uniquement la colonne 'ratio'
print("Contenu de la colonne 'ratio':")
df1.select('ratio').show()

Contenu de la colonne 'ratio':
+-----+
|ratio|
+-----+
|  2.0|
| NULL|
|  2.0|
| NULL|
|  2.0|
+-----+



3.  **Quel est le maximum de `rain_duration_9am` qui existe dans la base (utiliser `orderBy` et `head`)**

    Pour trouver la valeur maximale d'une colonne, nous pouvons trier le DataFrame par cette colonne en ordre décroissant et prendre le premier élément.

In [14]:
from pyspark.sql.functions import desc

# Trouver le maximum de 'rain_duration_9am'
max_rain_duration = df.orderBy(desc('rain_duration_9am')).select('rain_duration_9am').head()
print(f"Le maximum de rain_duration_9am est : {max_rain_duration['rain_duration_9am']}")

Le maximum de rain_duration_9am est : 20


4.  **Calculer le `mean` de la colonne « `rain_accumulation_9am` »**

    Nous utiliserons la fonction `mean` de `pyspark.sql.functions` pour calculer la moyenne.

In [15]:
from pyspark.sql.functions import mean, max, min

# Calculer la moyenne de 'rain_accumulation_9am'
mean_rain_accumulation = df.select(mean('rain_accumulation_9am')).collect()[0][0]
print(f"La moyenne de rain_accumulation_9am est : {mean_rain_accumulation}")

La moyenne de rain_accumulation_9am est : 3.5


5.  **De même calculer le `max` et le `min` de `rain_duration_9am`**

    Nous utiliserons les fonctions `max` et `min` de `pyspark.sql.functions`.

In [16]:
# Calculer le maximum de 'rain_duration_9am'
max_rain_duration_func = df.select(max('rain_duration_9am')).collect()[0][0]
print(f"Le maximum de rain_duration_9am (avec fonction max) est : {max_rain_duration_func}")

# Calculer le minimum de 'rain_duration_9am'
min_rain_duration = df.select(min('rain_duration_9am')).collect()[0][0]
print(f"Le minimum de rain_duration_9am est : {min_rain_duration}")

Le maximum de rain_duration_9am (avec fonction max) est : 20
Le minimum de rain_duration_9am est : 0


6.  **Combien de fois le `air_temp_9h` est > 70**

    Nous utiliserons la méthode `filter()` pour sélectionner les lignes qui satisfont la condition, puis `count()` pour obtenir le nombre de ces lignes.

In [17]:
# Compter les occurrences où 'air_temp_9am' est supérieur à 70
count_temp_gt_70 = df.filter(df['air_temp_9am'] > 70).count()
print(f"Le nombre de fois où air_temp_9am est supérieur à 70 est : {count_temp_gt_70}")

Le nombre de fois où air_temp_9am est supérieur à 70 est : 0


### Step 5 (Partie 1) : Suppression des lignes avec des missing values

La colonne `air_pressure_9am` est mentionnée comme pouvant contenir des valeurs manquantes, et nous avons également introduit des `NULL` dans la colonne `ratio`. Nous allons utiliser la méthode `dropna()` pour supprimer les lignes de notre DataFrame qui contiennent des valeurs manquantes. Nous allons créer un nouveau DataFrame pour ne pas modifier l'original (df).

Pour démontrer, nous allons d'abord montrer le nombre de lignes avant la suppression, puis après.

In [18]:
# Afficher le nombre de lignes avant la suppression
print(f"Nombre de lignes avant suppression des valeurs manquantes : {df1.count()}")

# Supprimer les lignes avec des valeurs manquantes (NaN ou NULL) de df1
# Nous utilisons df1 car c'est le DataFrame qui contient la colonne 'ratio' avec des NULL
df_cleaned = df1.dropna()

# Afficher le nombre de lignes après la suppression
print(f"Nombre de lignes après suppression des valeurs manquantes : {df_cleaned.count()}")

# Afficher le DataFrame nettoyé
print("DataFrame après suppression des lignes avec valeurs manquantes:")
df_cleaned.show()

Nombre de lignes avant suppression des valeurs manquantes : 5
Nombre de lignes après suppression des valeurs manquantes : 3
DataFrame après suppression des lignes avec valeurs manquantes:
+------------+-----------------+---------------------+----------------+-----+
|air_temp_9am|rain_duration_9am|rain_accumulation_9am|air_pressure_9am|ratio|
+------------+-----------------+---------------------+----------------+-----+
|        20.5|               10|                  5.0|          1012.5|  2.0|
|        18.3|               20|                 10.0|          1013.2|  2.0|
|        19.8|                5|                  2.5|          1014.1|  2.0|
+------------+-----------------+---------------------+----------------+-----+



### Step 5 (Partie 2) : Calculer la corrélation entre deux colonnes

**Pourquoi on doit calculer la corrélation entre les attributs dans un dataset ?**

Le calcul de la corrélation entre les attributs est une étape importante dans l'exploration des données car il permet de comprendre la relation linéaire entre deux variables. Une corrélation positive indique que les variables ont tendance à augmenter ou diminuer ensemble, tandis qu'une corrélation négative signifie que l'une a tendance à augmenter lorsque l'autre diminue. L'ampleur du coefficient de corrélation (entre -1 et 1) indique la force de cette relation.

Cela aide à :
*   **Identifier les redondances** : Si deux variables sont fortement corrélées, l'une peut être redondante par rapport à l'autre.
*   **Sélection de caractéristiques** : Pour la modélisation, on peut choisir les caractéristiques les plus pertinentes.
*   **Comprendre les relations sous-jacentes** : Découvrir des liens entre les phénomènes mesurés.

**Calculer la corrélation entre `rain_accumulation_9am` et `rain_duration_9am` :**

Nous allons utiliser la méthode `corr()` du DataFrame Spark pour calculer cette corrélation.

In [19]:
# Calculer la corrélation entre 'rain_accumulation_9am' et 'rain_duration_9am'
correlation = df_cleaned.corr('rain_accumulation_9am', 'rain_duration_9am')
print(f"La corrélation entre rain_accumulation_9am et rain_duration_9am est : {correlation}")

# Que peut-on conclure du résultat ?
# Un coefficient de corrélation proche de 1 indique une forte corrélation positive.
# Un coefficient proche de -1 indique une forte corrélation négative.
# Un coefficient proche de 0 indique une faible ou pas de corrélation linéaire.

if correlation > 0.7:
    conclusion = "Il y a une forte corrélation positive entre la durée et l'accumulation de pluie. Cela signifie que plus il pleut longtemps, plus l'accumulation de pluie est importante."
elif correlation < -0.7:
    conclusion = "Il y a une forte corrélation négative entre la durée et l'accumulation de pluie."
elif correlation > 0.3:
    conclusion = "Il y a une corrélation positive modérée entre la durée et l'accumulation de pluie."
elif correlation < -0.3:
    conclusion = "Il y a une corrélation négative modérée entre la durée et l'accumulation de pluie."
else:
    conclusion = "Il y a une faible ou pas de corrélation linéaire entre la durée et l'accumulation de pluie."

print(f"Conclusion : {conclusion}")

La corrélation entre rain_accumulation_9am et rain_duration_9am est : 1.0
Conclusion : Il y a une forte corrélation positive entre la durée et l'accumulation de pluie. Cela signifie que plus il pleut longtemps, plus l'accumulation de pluie est importante.


### Step 6 : Imputer les valeurs manquantes

Au lieu de supprimer les lignes contenant des valeurs manquantes, une autre approche courante est de les imputer, c'est-à-dire de les remplacer par une valeur estimée. Ici, nous allons remplacer les valeurs manquantes par la valeur moyenne de chaque colonne. Pour cela, nous allons d'abord importer la fonction `avg` et faire une copie du DataFrame original pour travailler dessus (`df_imputed`).

In [20]:
from pyspark.sql.functions import avg

# Faire une copie du DataFrame original pour l'imputation
df_imputed = df.alias("df_imputed_copy")

# Parcourir chaque colonne numérique du DataFrame pour calculer et imputer les valeurs moyennes
print("Imputation des valeurs manquantes par la moyenne de la colonne...")
for col_name in df_imputed.columns:
    # Pour les colonnes numériques (supposons qu'elles sont toutes numériques ici pour la démo)
    if df_imputed.schema[col_name].dataType.typeName() in ['double', 'integer', 'float']:
        mean_val = df_imputed.select(avg(col_name)).collect()[0][0]
        if mean_val is not None:
            df_imputed = df_imputed.na.fill({col_name: mean_val})
            print(f"  - Colonne '{col_name}': valeurs manquantes remplacées par la moyenne {mean_val:.2f}")
        else:
            print(f"  - Colonne '{col_name}': impossible de calculer la moyenne (toutes les valeurs sont nulles ou non numériques)")

print("DataFrame après imputation:")
df_imputed.show()

# Utiliser describe() pour comparer les statistiques de chaque colonne ; que remarquez-vous ?
print("Statistiques récapitulatives après imputation des valeurs manquantes:")
df_imputed.describe().show()

print("Statistiques récapitulatives du DataFrame original (pour comparaison):")
df.describe().show()

# Remarque : après imputation par la moyenne, les `count` pour les colonnes numériques devraient correspondre au nombre total de lignes,
# et les `mean` devraient rester les mêmes (ou très proches si le jeu de données est grand et que les NA étaient peu nombreux),
# mais les `stddev` (écart-type) pourraient diminuer car l'imputation réduit la variance.

Imputation des valeurs manquantes par la moyenne de la colonne...
  - Colonne 'air_temp_9am': valeurs manquantes remplacées par la moyenne 21.14
  - Colonne 'rain_duration_9am': valeurs manquantes remplacées par la moyenne 7.00
  - Colonne 'rain_accumulation_9am': valeurs manquantes remplacées par la moyenne 3.50
  - Colonne 'air_pressure_9am': valeurs manquantes remplacées par la moyenne 1012.16
DataFrame après imputation:
+------------+-----------------+---------------------+----------------+
|air_temp_9am|rain_duration_9am|rain_accumulation_9am|air_pressure_9am|
+------------+-----------------+---------------------+----------------+
|        20.5|               10|                  5.0|          1012.5|
|        22.0|                0|                  0.0|          1011.0|
|        18.3|               20|                 10.0|          1013.2|
|        25.1|                0|                  0.0|          1010.0|
|        19.8|                5|                  2.5|          1014

---

## Exercice d’application (par binome) : Création d'un modèle de classification

Le but de cet exercice est de créer un modèle de classification d’une base de données choisie par vous.

### Step 1 : Définir la dataset et l’explorer

Pour cet exercice, nous allons choisir un jeu de données public couramment utilisé pour la classification. Pour l'exemple, nous allons utiliser le jeu de données Iris, qui est un classique pour les tâches de classification et est souvent utilisé pour démontrer des modèles d'apprentissage automatique. Nous allons le simuler ou le charger si disponible.

Nous allons d'abord créer une `SparkSession` (si non déjà existante) et charger les données.

In [21]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Créer une SparkSession si elle n'existe pas déjà
spark = SparkSession.builder.appName("ClassificationExample").getOrCreate()

# Pour cet exemple, nous allons simuler le chargement d'un dataset simple comme Iris.
# Dans un cas réel, vous chargeriez votre propre fichier CSV ou autre format.
# Créons un DataFrame simple pour l'exemple de classification (similaire à Iris).
# Features: sepal_length, sepal_width, petal_length, petal_width
# Label: species (0, 1, 2)

data = [
    (5.1, 3.5, 1.4, 0.2, 0.0),  # Setosa
    (4.9, 3.0, 1.4, 0.2, 0.0),
    (6.3, 3.3, 6.0, 2.5, 2.0),  # Virginica
    (5.8, 2.7, 5.1, 1.9, 2.0),
    (7.0, 3.2, 4.7, 1.4, 1.0),  # Versicolor
    (6.4, 3.2, 4.5, 1.5, 1.0),
    (5.0, 3.4, 1.5, 0.2, 0.0),
    (5.9, 3.0, 5.1, 1.8, 2.0),
    (6.9, 3.1, 4.9, 1.5, 1.0),
    (4.4, 3.2, 1.3, 0.2, 0.0)
]

columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]

df_classification = spark.createDataFrame(data, columns)

print("Dataset de classification (Iris simulé) :")
df_classification.show()
df_classification.printSchema()
df_classification.describe().show()

Dataset de classification (Iris simulé) :
+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2|    0.0|
|         4.9|        3.0|         1.4|        0.2|    0.0|
|         6.3|        3.3|         6.0|        2.5|    2.0|
|         5.8|        2.7|         5.1|        1.9|    2.0|
|         7.0|        3.2|         4.7|        1.4|    1.0|
|         6.4|        3.2|         4.5|        1.5|    1.0|
|         5.0|        3.4|         1.5|        0.2|    0.0|
|         5.9|        3.0|         5.1|        1.8|    2.0|
|         6.9|        3.1|         4.9|        1.5|    1.0|
|         4.4|        3.2|         1.3|        0.2|    0.0|
+------------+-----------+------------+-----------+-------+

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double

### Step 2 : Faire le preprocessing nécessaire

Pour les algorithmes de Machine Learning de Spark ML, les colonnes de caractéristiques (features) doivent être combinées en une seule colonne de type `Vector`.

Nous utiliserons `VectorAssembler` pour transformer les colonnes numériques `sepal_length`, `sepal_width`, `petal_length`, `petal_width` en une seule colonne `features`. La colonne `species` sera notre colonne cible (label).

In [25]:
# Définir les colonnes de caractéristiques (features) et la colonne cible (label)
feature_columns = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
label_column = "species"

# Utiliser VectorAssembler pour combiner les colonnes de caractéristiques en une seule colonne 'features'
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features")

df_assembled = assembler.transform(df_classification)

# Afficher le DataFrame avec la nouvelle colonne 'features'
print("DataFrame après VectorAssembler:")
df_assembled.show(5)

# S'assurer que le type de la colonne label est correct pour la classification
# Pour LogisticRegression, le label doit être de type numérique.
# Notre colonne 'species' est déjà double, ce qui est suffisant.


DataFrame après VectorAssembler:
+------------+-----------+------------+-----------+-------+-----------------+
|sepal_length|sepal_width|petal_length|petal_width|species|         features|
+------------+-----------+------------+-----------+-------+-----------------+
|         5.1|        3.5|         1.4|        0.2|    0.0|[5.1,3.5,1.4,0.2]|
|         4.9|        3.0|         1.4|        0.2|    0.0|[4.9,3.0,1.4,0.2]|
|         6.3|        3.3|         6.0|        2.5|    2.0|[6.3,3.3,6.0,2.5]|
|         5.8|        2.7|         5.1|        1.9|    2.0|[5.8,2.7,5.1,1.9]|
|         7.0|        3.2|         4.7|        1.4|    1.0|[7.0,3.2,4.7,1.4]|
+------------+-----------+------------+-----------+-------+-----------------+
only showing top 5 rows


### Step 2 : Faire le preprocessing nécessaire

Pour les algorithmes de Machine Learning de Spark ML, les colonnes de caractéristiques (features) doivent être combinées en une seule colonne de type `Vector`.

Nous utiliserons `VectorAssembler` pour transformer les colonnes numériques `sepal_length`, `sepal_width`, `petal_length`, `petal_width` en une seule colonne `features`. La colonne `species` sera notre colonne cible (label).

In [22]:
# Définir les colonnes de caractéristiques (features) et la colonne cible (label)
feature_columns = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
label_column = "species"

# Utiliser VectorAssembler pour combiner les colonnes de caractéristiques en une seule colonne 'features'
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features")

df_assembled = assembler.transform(df_classification)

# Afficher le DataFrame avec la nouvelle colonne 'features'
print("DataFrame après VectorAssembler:")
df_assembled.show(5)

# S'assurer que le type de la colonne label est correct pour la classification
# Pour LogisticRegression, le label doit être de type numérique.
# Notre colonne 'species' est déjà double, ce qui est suffisant.


DataFrame après VectorAssembler:
+------------+-----------+------------+-----------+-------+-----------------+
|sepal_length|sepal_width|petal_length|petal_width|species|         features|
+------------+-----------+------------+-----------+-------+-----------------+
|         5.1|        3.5|         1.4|        0.2|    0.0|[5.1,3.5,1.4,0.2]|
|         4.9|        3.0|         1.4|        0.2|    0.0|[4.9,3.0,1.4,0.2]|
|         6.3|        3.3|         6.0|        2.5|    2.0|[6.3,3.3,6.0,2.5]|
|         5.8|        2.7|         5.1|        1.9|    2.0|[5.8,2.7,5.1,1.9]|
|         7.0|        3.2|         4.7|        1.4|    1.0|[7.0,3.2,4.7,1.4]|
+------------+-----------+------------+-----------+-------+-----------------+
only showing top 5 rows


### Step 3 : Créer votre modèle de classification et la prédiction

Nous allons maintenant entraîner un modèle de classification. Pour cet exemple, nous utiliserons la Régression Logistique (`LogisticRegression`) de PySpark ML. Ensuite, nous allons évaluer les performances de ce modèle.

Nous allons également diviser les données en ensembles d'entraînement et de test pour évaluer correctement le modèle sur des données qu'il n'a pas vues pendant l'entraînement.

In [23]:
# Diviser les données en ensembles d'entraînement et de test (par exemple, 70% pour l'entraînement, 30% pour le test)
(training_data, test_data) = df_assembled.randomSplit([0.7, 0.3], seed=42)

print(f"Taille de l'ensemble d'entraînement: {training_data.count()}")
print(f"Taille de l'ensemble de test: {test_data.count()}")

# Créer un modèle de régression logistique
lr = LogisticRegression(featuresCol='features', labelCol='species', maxIter=10)

# Entraîner le modèle sur les données d'entraînement
lr_model = lr.fit(training_data)

print("Modèle de régression logistique entraîné.")

# Faire des prédictions sur les données de test
predictions = lr_model.transform(test_data)

# Afficher les prédictions (quelques lignes)
print("Prédictions sur les données de test:")
predictions.select("features", "species", "prediction", "probability").show(5)

# Évaluer le modèle
evaluator = MulticlassClassificationEvaluator(
    labelCol="species", predictionCol="prediction", metricName="accuracy")

accuracy = evaluator.evaluate(predictions)
print(f"Précision (Accuracy) du modèle sur l'ensemble de test = {accuracy}")

Taille de l'ensemble d'entraînement: 8
Taille de l'ensemble de test: 2
Modèle de régression logistique entraîné.
Prédictions sur les données de test:
+-----------------+-------+----------+--------------------+
|         features|species|prediction|         probability|
+-----------------+-------+----------+--------------------+
|[5.8,2.7,5.1,1.9]|    2.0|       2.0|[1.28723908569553...|
|[4.4,3.2,1.3,0.2]|    0.0|       0.0|[0.99998939317002...|
+-----------------+-------+----------+--------------------+

Précision (Accuracy) du modèle sur l'ensemble de test = 1.0


---

## Exercice 2 : Système de recommandation avec PySpark

Cet exercice se concentre sur la construction d'un système de recommandation en utilisant l'algorithme Alternating Least Squares (ALS) de Spark MLlib.

### Step 1 : Créer un `SparkSession` et le démarrer

Nous allons d'abord nous assurer qu'une `SparkSession` est active ou en créer une nouvelle, ce qui est le point d'entrée pour utiliser Spark.

In [24]:
from pyspark.sql import SparkSession

# Créer une SparkSession si elle n'existe pas déjà
# appName est défini pour cette tâche spécifique
spark = SparkSession.builder.appName("RecommendationSystem").getOrCreate()

print("SparkSession est prête.")

SparkSession est prête.


### Step 2 : Importer l’algo ALS

Nous allons importer les classes nécessaires pour construire et évaluer notre modèle de système de recommandation : `ALS` pour l'algorithme de factorisation matricielle et `RegressionEvaluator` pour mesurer la performance du modèle.

In [26]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

print("Classes ALS et RegressionEvaluator importées.")

Classes ALS et RegressionEvaluator importées.


### Step 3 : Charger les données de « `rating.csv` » dans un dataframe `df`

Nous allons charger le fichier `rating.csv` dans un DataFrame Spark. Ce fichier devrait contenir au moins les colonnes `userId`, `movieId` et `rating` pour être utilisé par ALS. Comme précédemment, nous allons inclure une vérification et la création d'un fichier factice si `rating.csv` n'est pas trouvé.

In [27]:
import os

# NOTE: Assurez-vous que 'rating.csv' est disponible dans votre environnement Colab.
# Pour l'exemple, nous allons créer un fichier dummy si 'rating.csv' n'existe pas.
if not os.path.exists('rating.csv'):
    # Créer un fichier CSV dummy pour l'exemple de recommandation
    dummy_ratings_data = "userId,movieId,rating,timestamp\n1,1,5.0,100\n1,2,3.0,100\n1,3,4.0,100\n2,1,4.0,100\n2,2,5.0,100\n2,4,2.0,100\n3,1,3.0,100\n3,3,2.0,100\n4,2,4.0,100\n4,4,5.0,100\n"
    with open('rating.csv', 'w') as f:
        f.write(dummy_ratings_data)
    print("Fichier 'rating.csv' créé pour l'exemple.")

# Charger les données de 'rating.csv' dans un DataFrame
df_ratings = spark.read.csv("rating.csv", header=True, inferSchema=True)

print("DataFrame des ratings chargé avec succès.")
df_ratings.printSchema()
df_ratings.show(5)

Fichier 'rating.csv' créé pour l'exemple.
DataFrame des ratings chargé avec succès.
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   5.0|      100|
|     1|      2|   3.0|      100|
|     1|      3|   4.0|      100|
|     2|      1|   4.0|      100|
|     2|      2|   5.0|      100|
+------+-------+------+---------+
only showing top 5 rows


### Step 4 : Afficher le contenu de `df_ratings`

Nous allons afficher les premières lignes du DataFrame `df_ratings` pour nous assurer que les données ont été chargées correctement et pour visualiser leur format.

In [28]:
# Afficher le contenu du DataFrame df_ratings
print("Contenu du DataFrame df_ratings:")
df_ratings.show()

Contenu du DataFrame df_ratings:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   5.0|      100|
|     1|      2|   3.0|      100|
|     1|      3|   4.0|      100|
|     2|      1|   4.0|      100|
|     2|      2|   5.0|      100|
|     2|      4|   2.0|      100|
|     3|      1|   3.0|      100|
|     3|      3|   2.0|      100|
|     4|      2|   4.0|      100|
|     4|      4|   5.0|      100|
+------+-------+------+---------+



### Step 5 : Supprimer les colonnes inutilisables pour cet exercice

Pour l'algorithme ALS de Spark, nous avons besoin de colonnes pour l'utilisateur, l'élément et la notation. Dans notre jeu de données `rating.csv`, la colonne `timestamp` n'est pas nécessaire pour le modèle de recommandation de base et peut être supprimée.

In [29]:
# Supprimer la colonne 'timestamp'
df_ratings_cleaned = df_ratings.drop('timestamp')

print("DataFrame après suppression de la colonne 'timestamp':")
df_ratings_cleaned.show(5)
df_ratings_cleaned.printSchema()

DataFrame après suppression de la colonne 'timestamp':
+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|      1|   5.0|
|     1|      2|   3.0|
|     1|      3|   4.0|
|     2|      1|   4.0|
|     2|      2|   5.0|
+------+-------+------+
only showing top 5 rows
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)



### Étape 6 : Diviser la dataset en ensembles d'entraînement et de test

Pour préparer notre modèle de recommandation, nous devons diviser le `df_ratings_cleaned` en deux sous-ensembles : un ensemble d'entraînement (`training_ratings`) et un ensemble de test (`test_ratings`). Cela nous permettra d'entraîner le modèle sur une partie des données et d'évaluer ses performances sur des données non vues.

Nous allons utiliser une division aléatoire avec 80% des données pour l'entraînement et 20% pour le test.

In [30]:
# Diviser la dataset en training et testing sets (80% pour l'entraînement, 20% pour le test)
(training_ratings, test_ratings) = df_ratings_cleaned.randomSplit([0.8, 0.2], seed=42)

print(f"Nombre de lignes pour l'entraînement : {training_ratings.count()}")
print(f"Nombre de lignes pour le test : {test_ratings.count()}")

print("Premières lignes de l'ensemble d'entraînement:")
training_ratings.show(5)

print("Premières lignes de l'ensemble de test:")
test_ratings.show(5)

Nombre de lignes pour l'entraînement : 7
Nombre de lignes pour le test : 3
Premières lignes de l'ensemble d'entraînement:
+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|      1|   5.0|
|     1|      2|   3.0|
|     2|      1|   4.0|
|     2|      2|   5.0|
|     2|      4|   2.0|
+------+-------+------+
only showing top 5 rows
Premières lignes de l'ensemble de test:
+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|      3|   4.0|
|     3|      1|   3.0|
|     4|      2|   4.0|
+------+-------+------+



### Étape 7 : Entraîner le modèle ALS (Alternating Least Squares)

Maintenant que nous avons divisé notre jeu de données, nous pouvons entraîner le modèle de système de recommandation en utilisant l'algorithme ALS. ALS est couramment utilisé pour les systèmes de recommandation collaboratifs.

Nous allons configurer l'ALS en spécifiant les colonnes pour `userId`, `movieId` (l'élément) et `rating`. Nous utiliserons également `coldStartStrategy="drop"` pour gérer les cas où nous rencontrons de nouveaux utilisateurs ou éléments qui n'étaient pas présents dans les données d'entraînement, en les ignorant lors de la prédiction.

In [31]:
# Construire le modèle ALS
als = ALS(maxIter=5, regParam=0.09, userCol="userId", itemCol="movieId", ratingCol="rating",
         coldStartStrategy="drop")

# Entraîner le modèle sur l'ensemble d'entraînement
model = als.fit(training_ratings)

print("Modèle ALS entraîné avec succès.")

Modèle ALS entraîné avec succès.


### Étape 8 : Faire des prédictions et évaluer le modèle

Après avoir entraîné le modèle ALS, nous allons faire des prédictions sur l'ensemble de test. Ces prédictions nous donneront les notes estimées pour les paires utilisateur-élément présentes dans l'ensemble de test.

Ensuite, nous évaluerons la performance de notre modèle en comparant ces prédictions avec les notes réelles. Une métrique couramment utilisée pour évaluer les systèmes de recommandation est la Root Mean Squared Error (RMSE), qui mesure l'ampleur des erreurs de prédiction.

In [32]:
# Faire des prédictions sur l'ensemble de test
predictions = model.transform(test_ratings)

# Afficher les 5 premières prédictions
print("Prédictions sur l'ensemble de test (premières 5 lignes) :")
predictions.show(5)

# Évaluer le modèle en utilisant la RMSE (Root Mean Squared Error)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE) = {rmse}")

Prédictions sur l'ensemble de test (premières 5 lignes) :
+------+-------+------+-----------+
|userId|movieId|rating| prediction|
+------+-------+------+-----------+
|     1|      3|   4.0|0.093830116|
|     3|      1|   3.0| -0.5030354|
|     4|      2|   4.0|  1.9443443|
+------+-------+------+-----------+

Root Mean Squared Error (RMSE) = 3.2534669092656503


### Étape 9 : Générer des recommandations pour les utilisateurs

Une fois le modèle ALS entraîné, nous pouvons l'utiliser pour générer des recommandations. Nous allons générer les 5 meilleurs films recommandés pour chaque utilisateur présent dans le jeu de données d'entraînement. Ces recommandations sont basées sur les notes prédites par le modèle ALS.

In [33]:
# Générer les 5 meilleures recommandations pour chaque utilisateur
user_recs = model.recommendForAllUsers(5)

print("Recommandations pour tous les utilisateurs (premières lignes) :")
user_recs.show(5, truncate=False)

# Alternativement, vous pouvez générer des recommandations pour un utilisateur spécifique ou pour tous les éléments.
# Par exemple, pour un utilisateur (userId=1):
# specific_user_recs = user_recs.filter(user_recs['userId'] == 1).collect()
# if specific_user_recs:
#     print(f"Recommandations pour l'utilisateur 1: {specific_user_recs[0]['recommendations']}")

Recommandations pour tous les utilisateurs (premières lignes) :
+------+-------------------------------------------------------------------+
|userId|recommendations                                                    |
+------+-------------------------------------------------------------------+
|1     |[{1, 4.885603}, {2, 3.073739}, {3, 0.093830116}, {4, -1.4111089}]  |
|2     |[{2, 4.8497276}, {1, 4.0614347}, {4, 1.9991126}, {3, -0.54255426}] |
|3     |[{3, 1.9300306}, {1, -0.5030354}, {4, -0.8565978}, {2, -0.9584617}]|
|4     |[{4, 4.951006}, {2, 1.9443443}, {1, 0.044593424}, {3, -0.5998105}] |
+------+-------------------------------------------------------------------+

